<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Régression linéaire - RMSE public : 3.2696
</div>
<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des bilbiothèques
</div>

In [7]:
import pandas as pd
from sklearn.linear_model import LinearRegression

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des documents de données temporelles et de test
</div>

In [8]:
stations_toulouse = pd.read_parquet("stations_toulouse.parquet") # Données géographiques (Colonnes : 'station', 'longitude', 'latitude', 'station_id')
history_toulouse_train = pd.read_parquet("history_toulouse_train.parquet")
history_toulouse_test = pd.read_parquet("history_toulouse_test.parquet")

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Données du train
</div>
On transforme les indexe en colonnes pour plus de praticité

In [9]:
train = history_toulouse_train.reset_index()

Prise en compte de la date

In [10]:
train["annee"] = train["commit_at"].dt.year
train["mois"] = train["commit_at"].dt.month
train["jour"] = train["commit_at"].dt.dayofweek
train["heure"] = train["commit_at"].dt.hour
train["minute"] = train["commit_at"].dt.minute

Prise en compte de la géolocalisation

In [11]:
train = train.merge(stations_toulouse, on="station", how="left")

Prise en compte du nombre total de stations

In [12]:
train["total"] = train["bikes"] + train["stands"]

Données utiles pour l'entrainement du modèle

In [13]:
colonnes_train = ["bikes", "annee", "mois", "jour", "heure", "minute", "longitude", "latitude", "total"]

On extrait tous les indexe station et commit_at_index sous forme de liste

In [14]:
station_index = history_toulouse_test.index.get_level_values("station")
commit_at_index = history_toulouse_test.index.get_level_values("commit_at")

On crée un dataframe reliant (station, date) au nombre de vélos

In [15]:
date_velo = train.set_index(["station", "commit_at"])["bikes"]

On associe chaque nom de colonne à une valeur numérique

In [16]:
decalage = {
    "bikes_+15min": 15,
    "bikes_+30min": 30,
    "bikes_+45min": 45,
    "bikes_+60min": 60,
    "bikes_+75min": 75,
    "bikes_+90min": 90,
    "bikes_+105min": 105,
    "bikes_+120min": 120,
}

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Données du test
</div>
On récupère le nombre de vélos à l'instant zero du test

In [17]:
instant_zero = list(zip(station_index, commit_at_index))
bikes_train = history_toulouse_train["bikes"]
stands_train = history_toulouse_train["stands"]

On localise chaque index (station, date) pour trouver le nombre de vélos et d'emplacement vides à l'instant zero du test

In [18]:
bikes_zero = pd.Series([bikes_train.loc[instant] for instant in instant_zero], index=history_toulouse_test.index)
stands_zero = pd.Series([stands_train.loc[instant] for instant in instant_zero], index=history_toulouse_test.index)

On récupère la géolocalisation de chaque station du test <br>
On définit la colonne "station" comme étant l'indexe

In [19]:
stations_pos = stations_toulouse.set_index("station")

On localise chaque station pour trouver leur géolocalisation

In [20]:
longitude_test = [stations_pos.loc[station, "longitude"] for station in station_index]
latitude_test = [stations_pos.loc[station, "latitude"] for station in station_index]

On crée un dataframe avec toutes les données du test dont on aura besoin pour prédire. <br>
Il faut qu'il y ait les mêmes données que celles utilisées pour l'entrainement

In [21]:
donnees_test = pd.DataFrame({
    "bikes": bikes_zero,
    "annee": commit_at_index.year,
    "mois": commit_at_index.month,
    "jour": commit_at_index.dayofweek,
    "heure": commit_at_index.hour,
    "minute": commit_at_index.minute,
    "longitude": longitude_test,
    "latitude": latitude_test,
    "total": bikes_zero + stands_zero,
}, index=history_toulouse_test.index)

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Entrainement et prédictions
</div>
On entraine un modèle par décalage horaire et on le fait prédire

In [22]:
for col, h in decalage.items():
    print(h/1.2, "%")
    # On crée une copie du dataframe pour chaque décalage horaire
    avec_futur = train.copy()
    # On crée une colonne avec l'horaire à prédire
    avec_futur["commit_at_cible"] = avec_futur["commit_at"] + pd.Timedelta(minutes=h)
    # On trouve le nombre de vélos en prenant en compte le décalage horaire
    indexes_cibles = list(zip(avec_futur["station"], avec_futur["commit_at_cible"]))
    avec_futur["bikes_futur"] = [date_velo.get(indexe) for indexe in indexes_cibles]
    # On retire les lignes dans lesquelles le nombre de vélos n'est pas connu
    avec_futur = avec_futur.dropna(subset=["bikes_futur"])

    # Prédictions
    model = LinearRegression()
    # On renseigne les données d'entrainement et leur réponse
    X = avec_futur[colonnes_train]
    y = avec_futur["bikes_futur"]
    # On entraine le modèle
    model.fit(X, y)
    # On prédit et écrit les prédictions dans history_toulouse_test
    history_toulouse_test[col] = model.predict(donnees_test[colonnes_train]).round().astype(int)

12.5 %
25.0 %
37.5 %
50.0 %
62.5 %
75.0 %
87.5 %
100.0 %


<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Sauvegarde du résultat
</div>

In [23]:
history_toulouse_test.to_parquet("BA_prediction_101_RL.parquet", index=True)
history_toulouse_test

,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,9,9,9,9,9,9,8,8
00002 - LAFAYETTE,2026-07-31 23:45:00,9,9,9,9,9,9,9,8
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,18,17,17,17,17,16,16,16
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,12,12,12,11,11,11,11,11
00003 - POMME,2025-01-05 05:45:00,17,17,16,16,16,16,15,15
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,0,0,0,0,1,1,1,1
PASTEUR - LAVOISIER,2025-08-05 10:15:00,0,0,0,0,0,0,0,0
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,0,0,0,0,0,0,0,0
